# Model Development

## Objective

Develop machine learning models capable of predicting traffic event priority using historical event records and engineered operational features.

The resulting model will support proactive traffic management by identifying potentially high-impact events and assisting authorities in resource allocation decisions.

---

## Modeling Workflow

1. Feature Selection
2. Leakage Analysis
3. Data Encoding
4. Train-Test Split
5. Model Training
6. Model Evaluation
7. Feature Importance Analysis

###Section 1: Load Feature Engineered Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/feature_engineered_events.csv")

print(df.shape)
df.head()

(8171, 23)


,event_type,latitude,longitude,address,event_cause,requires_road_closure,start_datetime,authenticated,modified_datetime,description,...,created_date,police_station,zone,junction,hour,day_of_week,month,is_weekend,is_peak_hour,event_category
0,unplanned,13.040004,77.518099,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",vehicle_breakdown,False,2024-03-07 17:01:48.111000+00:00,yes,2024-03-07 19:35:47.871698+00:00,s m circle in coming man track,...,2024-03-07 17:03:51.164032+00:00,Peenya,Unknown,Unknown,17,Thursday,March,0,1,Incident
1,unplanned,12.921876,77.645158,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",vehicle_breakdown,False,2024-01-30 04:07:24.173000+00:00,yes,2024-01-30 04:17:46.828979+00:00,Starting problem,...,2024-01-30 04:08:22.954979+00:00,HSR Layout,Unknown,Unknown,4,Tuesday,January,0,0,Incident
2,unplanned,12.955622,77.585708,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",others,False,2023-11-11 06:18:03.343000+00:00,yes,2024-01-30 04:56:03.282003+00:00,ಊರ್ವಶಿ ಜಂಕ್ಷನ್ ನಲ್ಲಿ ಒಳಚರಂಡಿ ಚೇಂಬರ್ ಗೆ ಹೊಸದಾಗಿ...,...,2023-11-11 06:20:00.989398+00:00,Wilson Garden,Central Zone 2,UrvashiJunction,6,Saturday,November,1,0,Other
3,unplanned,13.006147,77.579435,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...",tree_fall,True,2024-03-07 17:56:55.061000+00:00,yes,2024-03-14 07:42:05.550050+00:00,tree fall,...,2024-03-07 17:58:56.696892+00:00,Sadashivanagar,Unknown,Unknown,17,Thursday,March,0,1,Natural
4,unplanned,12.953980,77.585233,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",vehicle_breakdown,False,2024-01-30 04:56:32.348000+00:00,yes,2024-01-30 05:35:17.339080+00:00,[LOCATION] ಪೈಪ್ [PERSON] ವಾಹನ ಆಫ್ ಆಗಿರುತ್ತದೆ ಸರ್,...,2024-01-30 04:58:55.937662+00:00,Wilson Garden,Unknown,LalbaghMainGateJunc,4,Tuesday,January,0,0,Incident


In [ ]:
df_model = df.copy()

In [ ]:
drop_cols = [
    'start_datetime',
    'created_date',
    'modified_datetime',
    'address',
    'description',
    'corridor'
]

df_model.drop(columns=drop_cols, inplace=True)

print(df_model.shape)

(8171, 17)


In [ ]:
df_model.columns.tolist()

['event_type',
 'latitude',
 'longitude',
 'event_cause',
 'requires_road_closure',
 'authenticated',
 'veh_type',
 'priority',
 'police_station',
 'zone',
 'junction',
 'hour',
 'day_of_week',
 'month',
 'is_weekend',
 'is_peak_hour',
 'event_category']

In [ ]:
pd.crosstab(
    df_model['authenticated'],
    df_model['priority'],
    normalize='index'
) * 100

priority,High,Low
authenticated,,
no,63.257200,36.742800
yes,61.320491,38.679509


In [ ]:
X = df_model.drop(columns=['priority'])

y = df_model['priority']

print(X.shape)
print(y.shape)

(8171, 16)
(8171,)


In [ ]:
y = y.map({
    'Low': 0,
    'High': 1
})

y.value_counts()

,count
priority,
1,5030
0,3141


In [ ]:
categorical_cols = [
    'event_type',
    'event_cause',
    'authenticated',
    'veh_type',
    'police_station',
    'zone',
    'junction',
    'day_of_week',
    'month',
    'event_category'
]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

print(X.shape)

(8171, 407)


In [ ]:
df_model['junction'].nunique()

295

In [ ]:
df_model['police_station'].nunique()

54

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (6536, 407)
X_test : (1635, 407)
y_train: (6536,)
y_test : (1635,)


In [ ]:
y.value_counts(normalize=True) * 100

,proportion
priority,
1,61.559173
0,38.440827


# Data Preparation for Modeling

## Class Distribution

The target variable (`priority`) exhibits a moderate class imbalance, with High priority events occurring more frequently than Low priority events.

To ensure representative evaluation, a stratified train-test split is used.

---

## Train-Test Split

- Training Set: 80%
- Testing Set: 20%
- Random State: 42
- Stratification: Enabled

This approach preserves class proportions across both datasets and enables reliable model evaluation.

###Random Forest

# Random Forest Baseline Model

## Why Random Forest?

Random Forest is chosen as the baseline model because:

- Handles mixed feature types well.
- Captures nonlinear relationships.
- Robust to noise and outliers.
- Provides feature importance for interpretability.

The model will establish a performance benchmark for subsequent experiments.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
y_pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.8917431192660551
Precision: 0.8717488789237668
Recall   : 0.9662027833001988
F1 Score : 0.9165487977369166


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.77      0.85       629
           1       0.87      0.97      0.92      1006

    accuracy                           0.89      1635
   macro avg       0.90      0.87      0.88      1635
weighted avg       0.90      0.89      0.89      1635



In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[486 143]
 [ 34 972]]


# Random Forest Baseline Results

## Model Performance

| Metric | Score |
|----------|----------|
| Accuracy | 89.17% |
| Precision | 87.17% |
| Recall | 96.62% |
| F1 Score | 91.65% |

---

## Confusion Matrix

| Actual / Predicted | Low | High |
|-------------------|-----|------|
| Low | 486 | 143 |
| High | 34 | 972 |

---

## Observation

The Random Forest model achieved an overall accuracy of **89.17%**, significantly outperforming the baseline accuracy of **61.56%**.

The model demonstrates particularly strong performance in identifying High Priority events, achieving a Recall of **96.62%**.

Only **34 High Priority events** were incorrectly classified as Low Priority.

---

## Insight

From an operational traffic-management perspective, missing a High Priority event is more costly than incorrectly flagging a Low Priority event as High.

The model shows:

- Very high sensitivity toward High Priority incidents.
- Strong overall classification capability.
- Good balance between Precision and Recall.

The elevated Recall score indicates that the system is effective at capturing potentially critical traffic events requiring intervention.

---

## Action

The Random Forest model establishes a strong baseline for EventPulse-AI.

Next steps:

1. Perform feature importance analysis.
2. Identify the most influential traffic-event attributes.
3. Compare performance against XGBoost.
4. Investigate potential overfitting through train-score comparison.
5. Build the congestion impact prediction pipeline on top of the classification engine.

In [ ]:
train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)

Train Accuracy: 1.0
Test Accuracy : 0.8917431192660551


In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
1,longitude,0.160374
0,latitude,0.120185
3,hour,0.048025
71,police_station_No Police Station,0.026865
61,police_station_K.R. Pura,0.019964
84,police_station_Wilson Garden,0.016793
77,police_station_Sheshadripuram,0.016283
59,police_station_Jnanabharathi,0.015942
57,police_station_Jayanagara,0.015515
372,junction_Unknown,0.013808


# Feature Importance Analysis

## Observation

Geospatial attributes such as latitude, longitude, police station, and junction emerged as the most influential predictors.

Temporal features such as hour of occurrence and peak-hour indicators also contributed significantly.

## Insight

The results suggest that historical event priority is strongly associated with specific traffic locations and operational zones.

This aligns with the objective of identifying recurring congestion-prone areas and supports the development of location-aware traffic management strategies.

## Action

A secondary experiment will be conducted by excluding location-related features to quantify their contribution to predictive performance and validate model robustness.

## Experiment 2: Location-Free Model

### Objective

Evaluate model performance after removing location-related features.

This experiment helps determine whether predictions are driven primarily by event characteristics or by geographic information.

Location-related attributes removed:

- Latitude
- Longitude
- Police Station
- Zone
- Junction

The resulting performance will be compared with the baseline Random Forest model.

In [ ]:
X_no_location = X.copy()

location_cols = [
    col for col in X_no_location.columns
    if (
        col == 'latitude'
        or col == 'longitude'
        or col.startswith('police_station_')
        or col.startswith('zone_')
        or col.startswith('junction_')
    )
]

print("Location Features Removed:", len(location_cols))

X_no_location = X_no_location.drop(
    columns=location_cols
)

print("New Shape:", X_no_location.shape)

Location Features Removed: 359
New Shape: (8171, 48)


In [ ]:
from sklearn.model_selection import train_test_split

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_no_location,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_no_location = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_no_location.fit(
    X_train2,
    y_train2
)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
y_pred2 = rf_no_location.predict(X_test2)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy :", accuracy_score(y_test2, y_pred2))
print("Precision:", precision_score(y_test2, y_pred2))
print("Recall   :", recall_score(y_test2, y_pred2))
print("F1 Score :", f1_score(y_test2, y_pred2))

Accuracy : 0.6250764525993884
Precision: 0.6801099908340972
Recall   : 0.7375745526838966
F1 Score : 0.7076776347162613


In [ ]:
train_acc2 = rf_no_location.score(
    X_train2,
    y_train2
)

test_acc2 = rf_no_location.score(
    X_test2,
    y_test2
)

print("Train Accuracy:", train_acc2)
print("Test Accuracy :", test_acc2)

Train Accuracy: 0.8953488372093024
Test Accuracy : 0.6250764525993884


In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Baseline RF': [
        0.8917,
        0.8717,
        0.9662,
        0.9165
    ],
    'No Location RF': [
        accuracy_score(y_test2, y_pred2),
        precision_score(y_test2, y_pred2),
        recall_score(y_test2, y_pred2),
        f1_score(y_test2, y_pred2)
    ]
})

comparison

,Metric,Baseline RF,No Location RF
0,Accuracy,0.8917,0.625076
1,Precision,0.8717,0.680110
2,Recall,0.9662,0.737575
3,F1 Score,0.9165,0.707678


## Experiment 2 Results: Location-Free Model

### Model Performance

| Metric | Baseline Random Forest | Location-Free Random Forest |
|----------|----------|----------|
| Accuracy | 89.17% | 62.51% |
| Precision | 87.17% | 68.01% |
| Recall | 96.62% | 73.76% |
| F1 Score | 91.65% | 70.77% |

---

### Observation

Removing geographic features resulted in a substantial decline across all evaluation metrics.

- Accuracy decreased from **89.17%** to **62.51%**
- Precision decreased from **87.17%** to **68.01%**
- Recall decreased from **96.62%** to **73.76%**
- F1 Score decreased from **91.65%** to **70.77%**

This performance drop highlights the significant contribution of location-based information in predicting traffic event priority.

---

### Insight

The experiment demonstrates that geographic attributes are the strongest predictors of event priority within the dataset.

Traffic events with similar causes can have vastly different operational impacts depending on where they occur. Major corridors, high-density intersections, and critical urban zones tend to experience greater disruption, making location a key factor in determining priority.

The results suggest that historical traffic patterns are highly location-dependent and that geospatial intelligence plays a crucial role in traffic impact forecasting.

---

### Business Implication

This finding validates the core hypothesis behind **EventPulse-AI**: effective traffic forecasting requires location-aware intelligence.

By incorporating geospatial information, authorities can:

- Identify recurring congestion hotspots.
- Prioritize resource deployment more effectively.
- Improve manpower and barricade allocation.
- Implement proactive traffic diversion strategies.

The experiment confirms that hotspot-aware forecasting can significantly enhance operational decision-making and traffic management outcomes.

---

### Conclusion

The comparison between the baseline and location-free models reveals that location-related features are responsible for a substantial portion of the predictive capability of the system.

Therefore, geospatial intelligence will remain a core component of the EventPulse-AI prediction framework and will be leveraged in subsequent impact forecasting and recommendation modules.

###Tuned Random Forest

## Experiment 3: Tuned Random Forest

### Objective

Reduce model overfitting and improve generalization performance by constraining tree complexity and introducing regularization parameters.

The tuned model will be compared against the baseline Random Forest to determine whether improved robustness can be achieved without sacrificing predictive performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, min_samples_leaf=5, min_samples_split=10,
                       n_estimators=300, n_jobs=-1, random_state=42)

In [ ]:
y_pred_tuned = rf_tuned.predict(X_test)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy :", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall   :", recall_score(y_test, y_pred_tuned))
print("F1 Score :", f1_score(y_test, y_pred_tuned))

Accuracy : 0.7944954128440367
Precision: 0.7641955835962145
Recall   : 0.963220675944334
F1 Score : 0.8522427440633246


In [ ]:
print("Train Accuracy:", rf_tuned.score(X_train, y_train))
print("Test Accuracy :", rf_tuned.score(X_test, y_test))

Train Accuracy: 0.8128824969400245
Test Accuracy : 0.7944954128440367


## Experiment 3 Results: Tuned Random Forest

### Model Performance

| Metric | Baseline RF | Tuned RF |
|----------|----------|----------|
| Accuracy | 89.17% | 79.45% |
| Precision | 87.17% | 76.42% |
| Recall | 96.62% | 96.32% |
| F1 Score | 91.65% | 85.22% |

---

### Observation

The tuned Random Forest substantially reduced overfitting by lowering the train-test performance gap.

However, overall predictive performance decreased across all evaluation metrics.

---

### Insight

While the tuned model generalized more consistently, the reduction in model complexity resulted in underfitting.

The baseline Random Forest retained superior predictive capability while maintaining strong generalization performance on unseen data.

---

### Conclusion

The baseline Random Forest was selected as the final model due to its higher Accuracy, F1 Score, and Recall, making it more suitable for operational traffic-priority forecasting.